# 🎥 Video Crash Analysis — Walkthrough Demo

This notebook demonstrates the end-to-end pipeline for analysing dashcam footage
using **Gemini 2.5 Pro** via **LiteLLM**. It is a standalone PoC that bypasses
Langfuse tracing and uses a local prompt instead.

**Pipeline steps:**
1. Configure LiteLLM connection (hardcoded for demo)
2. Load & validate a sample dashcam video
3. Encode the video to base64 and build the multimodal payload
4. Send the request to Gemini 2.5 Pro for crash-event detection
5. Parse the structured JSON response into Pydantic models
6. Display and explore the results

## 1. Setup & Configuration

In [ ]:
import litellm
import json
import base64
import logging
from pathlib import Path
from IPython.display import display, JSON, Video

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
# ── LiteLLM credentials ─────────────────────────────────────────────

LITELLM_API_KEY = "sk-jijtUNjqiqtTr1DSQUeKMQ"
LITELLM_BASE_URL = "https://genai-lounge-nx-litellm-uat-emea.zurich.com"

litellm.api_key = LITELLM_API_KEY
litellm.api_base = LITELLM_BASE_URL
litellm.suppress_debug_info = True

# Model identifier — routed through LiteLLM to the Gemini 2.5 Pro deployment
MODEL_NAME = "gemini/gemini-2.5-pro"

print(f" LiteLLM configured — base: {LITELLM_BASE_URL}")
print(f"   Model: {MODEL_NAME}")

## 2. Load & Validate the Sample Video

In [ ]:
# ── Point this to your local dashcam sample ─────────────────────────
VIDEO_PATH = Path("C:\\Users\\AMIR.RAHIMI\\Downloads\\Dashcam\\Inbound_CHO_Dashcam_20-50-780086.mp4")

# Supported formats and size cap (mirrors VideoAgentTemplate defaults)
SUPPORTED_FORMATS = {"mp4", "avi", "mov", "mkv", "webm"}
MAX_VIDEO_SIZE_MB = 100.0

VIDEO_MIME_TYPES = {
    "mp4": "video/mp4",
    "avi": "video/x-msvideo",
    "mov": "video/quicktime",
    "mkv": "video/x-matroska",
    "webm": "video/webm",
}

In [ ]:
def validate_video(path: Path) -> None:
    """Validate that the video exists, has a supported format, and is within size limits."""
    if not path.exists():
        raise FileNotFoundError(f"Video file not found: {path}")

    ext = path.suffix.lstrip(".").lower()
    if ext not in SUPPORTED_FORMATS:
        raise ValueError(f"Unsupported format '.{ext}'. Supported: {SUPPORTED_FORMATS}")

    size_mb = path.stat().st_size / (1024 * 1024)
    if size_mb > MAX_VIDEO_SIZE_MB:
        raise ValueError(f"Video is {size_mb:.1f} MB, exceeding limit of {MAX_VIDEO_SIZE_MB} MB.")

    print(f" Video validated: {path.name}  ({size_mb:.2f} MB, format: .{ext})")


validate_video(VIDEO_PATH)

In [ ]:
# ── Preview the video inline (works for .mp4 in JupyterLab) ────────
try:
    display(Video(str(VIDEO_PATH), embed=True, width=640))
except Exception as e:
    print(f"  Inline preview not available: {e}")
    print(f"   Open the file manually: {VIDEO_PATH.resolve()}")

## 3. Encode Video & Build Multimodal Payload

In [ ]:
def encode_video_to_base64(path: Path) -> str:
    """Read a video file and return its base64-encoded string."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def get_mime_type(path: Path) -> str:
    ext = path.suffix.lstrip(".").lower()
    return VIDEO_MIME_TYPES.get(ext, "video/mp4")


logger.info("Encoding video to base64 …")
video_b64 = encode_video_to_base64(VIDEO_PATH)
video_mime = get_mime_type(VIDEO_PATH)

print(f" Encoded: {len(video_b64) / (1024*1024):.2f} MB base64 string  (MIME: {video_mime})")

In [ ]:
# ── Build the multimodal content parts ──────────────────────────────
# This mirrors VideoLLM._build_video_content_parts()

CAMERA_TYPE = "dashcam_front"

video_content_parts = [
    {
        "type": "text",
        "text": f"[{CAMERA_TYPE}] Video file: {VIDEO_PATH.name}",
    },
    {
        "type": "image_url",
        "image_url": {
            "url": f"data:{video_mime};base64,{video_b64}",
        },
    },
]

print(f" Built {len(video_content_parts)} content parts for the multimodal payload.")

## 4. Craft the Prompt & Send to Gemini 2.5 Pro

In [ ]:
# ── System prompt (mirrors the hardcoded prompt in VideoLLM) ────────
SYSTEM_PROMPT = (
    "You are an expert video analyst specialising in road traffic "
    "collision detection. Analyse the provided dashcam and CCTV footage "
    "for crash events, near-misses, impact moments, sequence of events, "
    "road-user involvement (vehicles, pedestrians, cyclists), probable "
    "cause, and confidence per event. Include timestamps and concise "
    "descriptions, then return your findings as a JSON object."
)

# ── Task prompt (replaces the Langfuse-managed template) ────────────
TASK_PROMPT = """\
Analyse the attached dashcam video for any road traffic collision or
near-miss events. For each detected event, return a JSON object with
the following schema:

{
  "crash_events": [
    {
      "camera_type": "<dashcam_front | dashcam_rear | cctv>",
      "video_file_name": "<filename>",
      "crash_timestamp_seconds": <float>,
      "crash_timestamp_display": "<HH:MM:SS>",
      "confidence": <float 0-1>,
      "description": "<brief description of what happened>"
    }
  ],
  "summary": "<overall summary of the footage analysis>"
}

If no crash or near-miss events are detected, return an empty
"crash_events" list and state that in the summary.

Important:
- Provide timestamps relative to the start of the video.
- Be conservative with confidence scores.
- Distinguish between actual collisions and near-misses.
- Note all road users involved (vehicles, pedestrians, cyclists).
- Return ONLY the JSON object, no markdown fences or commentary.
"""

print("System prompt length:", len(SYSTEM_PROMPT), "chars")
print("Task prompt length:  ", len(TASK_PROMPT), "chars")

In [ ]:
# ── Assemble the full messages payload ────────────────────────────
# Append the task prompt as the final text part in the user message,
# matching the pattern used in VideoLLM.analyse_crash_footage()

user_content = video_content_parts + [
    {"type": "text", "text": TASK_PROMPT}
]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user",   "content": user_content},
]

print(f" Messages payload assembled — {len(messages)} messages, "
      f"{len(user_content)} user content parts.")

In [ ]:
# ── Call Gemini 2.5 Pro via LiteLLM ─────────────────────────────────
import time as _time

TEMPERATURE = 0.0


logger.info("Sending multimodal request to %s …", MODEL_NAME)

_start = _time.perf_counter()

raw_response = await litellm.acompletion(
    model=MODEL_NAME,
    messages=messages,
    temperature=TEMPERATURE,
    response_format={"type": "json_object"},
    cache={"no-store": True},
)

_elapsed = _time.perf_counter() - _start
logger.info("Response received — model: %s (%.2f seconds)", getattr(raw_response, "model", "unknown"), _elapsed)

## 5. Parse the Response into Pydantic Models

In [ ]:
# ── Import the project's Pydantic schemas ──────────────────────────
# If running outside the project structure, the models are defined
# inline below as a fallback.
import sys
from pathlib import Path

from pymongo.auth import TYPE_CHECKING
sys.path.insert(0, str(Path(".").resolve().parent))

_USE_FALLBACK = False

if TYPE_CHECKING:
    from video_analysis_poc.video_llm.schema.video import (
        CrashEvent,
        VideoAnalysisOutput,
    )
else:

    try:
        from video_analysis_poc.video_llm.schema.video import ( 
            CrashEvent,
            VideoAnalysisOutput,
        )
        print(" Imported schemas from video_analysis_poc")
    except ImportError:
        _USE_FALLBACK = True
        
    if _USE_FALLBACK:
        from pydantic import BaseModel, Field
        from typing import Literal
        
        class CrashEvent(BaseModel): 
            camera_type: str
            video_file_name: str
            crash_timestamp_seconds: float
            crash_timestamp_display: str
            confidence: float = Field(ge=0.0, le=1.0)
            description: str | None = None

        class VideoAnalysisOutput(BaseModel):
            task_name: str | None = None
            crash_events: list[CrashEvent] = Field(default_factory=list)
            summary: str | None = None
            raw_output: dict | None = None

        print("  Using inline fallback schema definitions.")

In [ ]:
# ── Extract and parse the LLM response ─────────────────────────────
from litellm.types.utils import ModelResponse, Choices

if not isinstance(raw_response, ModelResponse):
    raise TypeError(
        f"Expected ModelResponse, got {type(raw_response).__name__}. "
        "Streaming responses are not supported in this demo."
    )

choice = raw_response.choices[0]
if not isinstance(choice, Choices):
    raise TypeError(
        f"Expected Choices, got {type(choice).__name__}. "
        "Streaming responses are not supported in this demo."
    )

chat_content = choice.message.content
if not chat_content:
    raise ValueError("LLM returned empty content.")

# Strip markdown fences if the model wrapped the JSON
clean_content = chat_content.strip()
if clean_content.startswith("```"):
    clean_content = clean_content.split("\n", 1)[1]
    if clean_content.endswith("```"):
        clean_content = clean_content[:-3].strip()

parsed = json.loads(clean_content)
print(" JSON parsed successfully.")
print(f"   Keys: {list(parsed.keys())}")

In [ ]:
# ── Build the structured output ────────────────────────────────────
crash_events = [CrashEvent(**event) for event in parsed.get("crash_events", [])]

analysis_output = VideoAnalysisOutput(
    task_name="video_crash_analysis_demo",
    crash_events=crash_events,
    summary=parsed.get("summary"),
    raw_output=parsed,
)

print(f"\n Detected {len(analysis_output.crash_events)} crash event(s).")
print(f" Summary: {analysis_output.summary}")

## 6. Explore the Results

In [ ]:
# ── Display each crash event ───────────────────────────────────────
if analysis_output.crash_events:
    for i, event in enumerate(analysis_output.crash_events, 1):
        print(f"\n{'='*60}")
        print(f"  Event {i}")
        print(f"{'='*60}")
        print(f"  Camera:      {event.camera_type}")
        print(f"  File:        {event.video_file_name}")
        print(f"  Timestamp:   {event.crash_timestamp_display} ({event.crash_timestamp_seconds:.1f}s)")
        print(f"  Confidence:  {event.confidence:.2%}")
        print(f"  Description: {event.description}")
else:
    print("\n No crash events detected in the footage.")

In [ ]:
# ── Tabular view with pandas (optional) ────────────────────────────
import pandas as pd

if analysis_output.crash_events:
    df = pd.DataFrame([e.model_dump() for e in analysis_output.crash_events])
    display(df.style.format({"confidence": "{:.2%}"}).set_caption("Detected Crash Events"))
else:
    print("No events to display.")

In [ ]:
# ── Raw JSON output for inspection ─────────────────────────────────
display(JSON(analysis_output.raw_output, root="raw_model_output"))

In [ ]:
# ── Token usage summary ────────────────────────────────────────────
usage = getattr(raw_response, "usage", None)
if usage:
    print(" Token Usage")
    print(f"   Prompt tokens:     {getattr(usage, 'prompt_tokens', 'N/A')}")
    print(f"   Completion tokens: {getattr(usage, 'completion_tokens', 'N/A')}")
    print(f"   Total tokens:      {getattr(usage, 'total_tokens', 'N/A')}")
else:
    print("  Token usage not available in response.")

In [ ]:
# ── Save results to disk ───────────────────────────────────────────
output_path = Path("./crash_analysis_results.json")
output_path.write_text(json.dumps(analysis_output.model_dump(), indent=2, default=str))
print(f" Results saved to {output_path.resolve()}")

---

### Notes

- **Langfuse integration**: In production, prompts are managed via Langfuse templates
  (see `connection/langfuse/prompts.py`). This notebook uses a local prompt for
  standalone execution without a Langfuse server dependency.
- **Multiple cameras**: To analyse multiple sources, extend the `video_content_parts`
  list with additional video entries (one text label + one `image_url` per camera).
- **Model selection**: The `available_models.txt` in the repo lists all models
  available via the LiteLLM proxy. Gemini 2.5 Pro is the primary target for
  video-native multimodal inference.